# 02 · Fitur
Membangun semua blok fitur (F1-F8, FinBERT, embedding harian) dan menyimpannya. Scaler/PCA **tidak** di sini, tetapi di dalam pipeline CV.

In [1]:
import sys; sys.path.insert(0, "../src")
import json, joblib
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import t3
from t3 import CFG, OUT, PRICE, SEED
from eval_utils import *

In [2]:
prices, articles, sent = t3.load_export()
train_end = prices.loc[prices.split == "train", "date"].max()
X, blocks = t3.build_features(prices, articles, sent, train_end)
assert np.isfinite(X.select_dtypes("number")).all().all() and X.notna().all().all() and X.index.equals(prices.index)
X.to_parquet(OUT / "features.parquet")
(OUT / "blocks.json").write_text(json.dumps(blocks))
print(X.shape)
pd.Series({k: len(v) for k, v in blocks.items()}, name="n_kolom")

(1194, 1207)


harga              7
sent_lama         16
finbert            4
F1                 3
F2                 3
F3                 2
F4                 3
F5                 2
F8                 1
emb_bge_small    384
F7_bge_small       4
emb_bge_base     768
F7_bge_base        4
F6                 2
Name: n_kolom, dtype: int64

In [3]:
# Multikolinearitas antar blok (selain embedding): pasangan kolom beda blok dengan |corr| tertinggi
cols = {c: b for b, cs in blocks.items() if not b.startswith("emb_") for c in cs}
C = X[list(cols)].corr().abs()
pairs = C.where(np.triu(np.ones(C.shape), 1) > 0).stack().sort_values(ascending=False)
pairs[[cols[a] != cols[b] for a, b in pairs.index]].head(10).round(3)

anchor1_bge_small  anchor1_bge_base     0.994
anchor0_bge_small  anchor0_bge_base     0.993
anchor2_bge_small  anchor2_bge_base     0.993
novelty_bge_small  novelty_bge_base     0.986
anchor2_bge_small  anchor1_bge_base     0.983
anchor1_bge_small  anchor0_bge_base     0.975
                   anchor2_bge_base     0.975
anchor0_bge_small  anchor1_bge_base     0.969
anchor2_bge_small  anchor0_bge_base     0.963
has_news           anchor1_bge_small    0.954
dtype: float64